In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Define column names based on the Wisconsin Breast Cancer (Original) dataset
column_names = ['id', 'clump_thickness', 'uniformity_cell_size', 'uniformity_cell_shape',
                'marginal_adhesion', 'single_epithelial_cell_size', 'bare_nuclei',
                'bland_chromatin', 'normal_nucleoli', 'mitoses', 'diagnosis']

# Load dataset with no header and assign column names
df = pd.read_csv("../../datasets/BreastCancer/BreastCancerWc.csv", header=None, names=column_names)

# Data Cleaning
df.replace("?", np.nan, inplace=True)
df.dropna(inplace=True)

# Remove negative values
num_cols = ['clump_thickness', 'uniformity_cell_size',
       'uniformity_cell_shape', 'marginal_adhesion',
       'single_epithelial_cell_size', 'bland_chromatin', 'normal_nucleoli',
       'mitoses']
df = df[(df[num_cols] >= 0).all(axis=1)]

# Data Transformation
df['diagnosis'] = df['diagnosis'].map({2:0, 4:1})

# Outlier Removal using IQR method
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.5 * IQR
    lower_bound = Q1 - 1.5 * IQR

    df[col] = np.where(
        df[col] < lower_bound,
        lower_bound,
        np.where(
            df[col] > upper_bound,
            upper_bound,
            df[col]
        )
    )

X = df.drop(['id', 'diagnosis'], axis=1)
y = df['diagnosis']

# Feature Scaling
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Logistic Regression
lr = LogisticRegression(max_iter=3000)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

# Naive Bayes
nb = GaussianNB()
nb.fit(X_train, y_train)
nb_pred = nb.predict(X_test)

# Accuracy
print("Logistic Regression Accuracy:",
      accuracy_score(y_test, lr_pred))

print("Naive Bayes Accuracy:",
      accuracy_score(y_test, nb_pred))

# Reports
print("\nLogistic Regression Report:\n",
      classification_report(y_test, lr_pred))

print("\nNaive Bayes Report:\n",
      classification_report(y_test, nb_pred))

Logistic Regression Accuracy: 0.948905109489051
Naive Bayes Accuracy: 0.9708029197080292

Logistic Regression Report:
               precision    recall  f1-score   support

           0       0.93      0.99      0.96        79
           1       0.98      0.90      0.94        58

    accuracy                           0.95       137
   macro avg       0.95      0.94      0.95       137
weighted avg       0.95      0.95      0.95       137


Naive Bayes Report:
               precision    recall  f1-score   support

           0       0.97      0.97      0.97        79
           1       0.97      0.97      0.97        58

    accuracy                           0.97       137
   macro avg       0.97      0.97      0.97       137
weighted avg       0.97      0.97      0.97       137

